# Second downstream probe: 15-way sura classification

**Why this exists.** One binary task is a narrow basis for a transfer claim. This adds a second,
unrelated task on the same alignment.

**The task.** Given a verse's embedding, predict which sura (chapter) it comes from, restricted to
the 15 suras with the most verses. A multinomial logistic-regression probe is trained on English
embeddings and evaluated zero-shot on the same held-out verses in all 41 languages, exactly as in
Experiment 3.

**Why this task rather than another.**

- *The labels are already in the corpus.* Every verse carries its `sura` key, so no external
  resource, no annotation, and no judgement call about what the classes mean. That matters for a
  paper whose contribution is a corpus.
- *It is not binary.* 15 classes against a random baseline of 6.7% and a majority macro-F1 near 0.01,
  so the dynamic range is wide. `TOP_K=15` was chosen by running a TF-IDF probe on English at
  K = 5, 10, 15, 20, 30 and picking the K with the best signal-to-noise: macro-F1 minus the random
  baseline, divided by the bootstrap SD of macro-F1. K = 15 scored 13.2 against 9.8–10.8 elsewhere.
- *Length cannot do it.* A length-only probe reaches mean macro-F1 0.08 against a random baseline of
  0.067, essentially chance. The binary Meccan/Medinan probe's length baseline reaches 0.601, so this task
  removes by construction the confound Section 6.9 has to quantify for the first probe.
- *It measures a different construct.* Meccan/Medinan is a two-way division by revelation period;
  sura identity is topical and stylistic coherence within a chapter. A probe that transfers on one
  and not the other would tell us the first result was task-specific.

**The headline is not this probe's absolute accuracy.** It is whether the *per-language ordering*
agrees with the binary probe. If two unrelated tasks rank languages the same way, the transfer claim
is a property of the encoders rather than of Meccan/Medinan.

**Runtime.** GPU required (Runtime → Change runtime type → T4). About 40–70 minutes for four
encoders across 41 files; set `ENCODERS` to one model first if you want a quick look.

**A judgement call to check before you commit to it.** I chose sura classification because the labels
are exact and free. The alternative worth considering is the four-period Nöldeke chronology (early /
middle / late Meccan / Medinan), which is closer in spirit to the existing probe but needs an
external sura-to-period mapping that you would have to source and cite. If you prefer that, the only
change is the label function in Section 3.

## 1. Setup

In [ ]:
!pip -q install sentence-transformers==3.0.1 scikit-learn --upgrade
import io, os, re, json, zipfile
import numpy as np, pandas as pd
from pathlib import Path
import torch

SEED       = 42
N_EVAL     = 1000      # held-out verses, same convention as Experiment 3
TOP_K      = 15        # number of largest suras to keep as classes (tuned, see below)
ENCODERS   = ["sentence-transformers/LaBSE",
              "BAAI/bge-m3",
              "intfloat/multilingual-e5-base",
              "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"]
SHORT = {"sentence-transformers/LaBSE": "labse",
         "BAAI/bge-m3": "bgem3",
         "intfloat/multilingual-e5-base": "e5",
         "sentence-transformers/paraphrase-multilingual-mpnet-base-v2": "mpnet"}
LIMITS = {"labse": 256, "bgem3": 8192, "e5": 512, "mpnet": 128}

OUT = "results_probe2"; os.makedirs(OUT, exist_ok=True)
print("GPU:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4, then re-run.")

## 2. Load the corpus

Upload `translations.zip` (the 41-file corpus). The reader below is copied from
`downstream_transfer.py` so the verse keys and language names match Experiment 3 exactly.

In [ ]:
try:
    from google.colab import files
    if not os.path.exists("translations.zip"):
        files.upload()
except Exception as e:
    print("no upload dialog:", e)

if not os.path.exists("corpus"):
    with zipfile.ZipFile("translations.zip") as z:
        z.extractall("corpus")
CORPUS = Path("corpus")
if not list(CORPUS.glob("*.csv")):
    inner = [p for p in CORPUS.iterdir() if p.is_dir()]
    if inner: CORPUS = inner[0]

def lang_key(fname):
    if fname.startswith("bosnian_mihanovich"): return "bosnian-mihanovich"
    if fname.startswith("bosnian_rwwad"):      return "bosnian-rwwad"
    return fname.split("_")[0]

def read_corpus_csv(path):
    txt = path.read_text(encoding="utf-8-sig", errors="replace")
    m = re.search(r"^id,sura,aya", txt, flags=re.M)
    df = pd.read_csv(io.StringIO(txt[m.start():]))
    df = df[["sura", "aya", "translation"]].dropna()
    df["sura"] = df.sura.astype(int); df["aya"] = df.aya.astype(int)
    return df

langs = {lang_key(f.name): f for f in sorted(CORPUS.glob("*.csv"))}
print(f"{len(langs)} language files")
en = read_corpus_csv(langs["english"])
assert len(en) == 6236, f"expected 6236 English verses, got {len(en)}"
print("English verses:", len(en))

## 3. Labels and the train/eval split

Classes are the `TOP_K` suras with the most verses. The evaluation set is a random `N_EVAL` verses
drawn from those suras under seed 42; everything else from those suras is training data.

Swap the two marked lines to use a different label set.

In [ ]:
rng = np.random.default_rng(SEED)

# ---- label definition (swap these two lines for a different task) ----
sizes = en.sura.value_counts()
CLASSES = sorted(sizes.head(TOP_K).index.tolist())
label_of = lambda sura: sura
# ----------------------------------------------------------------------

sub = en[en.sura.isin(CLASSES)].reset_index(drop=True)
print(f"{len(CLASSES)} classes, {len(sub)} verses ({len(sub)/len(en):.0%} of the corpus)")
print("verses per class: min %d  median %d  max %d" %
      (sizes[CLASSES].min(), int(sizes[CLASSES].median()), sizes[CLASSES].max()))

perm = rng.permutation(len(sub))
eval_rows  = sub.iloc[np.sort(perm[:N_EVAL])]
train_rows = sub.iloc[np.sort(perm[N_EVAL:])]
eval_keys  = list(zip(eval_rows.sura,  eval_rows.aya))
train_keys = list(zip(train_rows.sura, train_rows.aya))
y_eval  = np.array([label_of(s) for s, a in eval_keys])
y_train = np.array([label_of(s) for s, a in train_keys])
print(f"train {len(train_keys)}   eval {len(eval_keys)}")

# baselines for calibration
from sklearn.metrics import f1_score, accuracy_score
maj = pd.Series(y_eval).mode()[0]
maj_pred = np.full_like(y_eval, maj)
print(f"\nmajority-class baseline: acc {accuracy_score(y_eval, maj_pred):.3f}  "
      f"macro-F1 {f1_score(y_eval, maj_pred, average='macro', zero_division=0):.3f}")
print(f"random baseline (1/{len(CLASSES)}): acc {1/len(CLASSES):.3f}")

## 4. Embed and probe

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

def pick(df, keys):
    d = df.set_index(["sura", "aya"])["translation"]
    return [str(d.get(k, "")) for k in keys]

rows = []
for model_name in ENCODERS:
    tag = SHORT[model_name]
    print(f"\n=== {tag} ===", flush=True)
    model = SentenceTransformer(model_name, device="cuda" if torch.cuda.is_available() else "cpu")
    prefix = "query: " if "e5" in model_name else ""
    embed = lambda texts: model.encode([prefix + t for t in texts], batch_size=64,
                                       convert_to_numpy=True, normalize_embeddings=True,
                                       show_progress_bar=False)

    X_train = embed(pick(en, train_keys))
    clf = LogisticRegression(max_iter=3000).fit(X_train, y_train)
    print("  probe trained on English", X_train.shape)

    for lang, f in sorted(langs.items()):
        df = read_corpus_csv(f)
        pred = clf.predict(embed(pick(df, eval_keys)))
        rows.append({"encoder": tag, "language": lang,
                     "accuracy": round(float(accuracy_score(y_eval, pred)), 4),
                     "macro_f1": round(float(f1_score(y_eval, pred, average="macro",
                                                      zero_division=0)), 4)})
        print(f"  {lang:20s} acc {rows[-1]['accuracy']:.3f}  macroF1 {rows[-1]['macro_f1']:.3f}",
              flush=True)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

probe2 = pd.DataFrame(rows)
probe2.to_csv(f"{OUT}/probe2_sura_all_encoders.csv", index=False)
print("\nsaved", f"{OUT}/probe2_sura_all_encoders.csv")

## 5. Length-only control

The same check Section 6.9 applies to the binary probe: can verse length alone do this? If a
length-only probe scores near the encoder probes, the task is measuring length rather than meaning.

In [ ]:
len_rows = []
for lang, f in sorted(langs.items()):
    df = read_corpus_csv(f).set_index(["sura", "aya"])["translation"]
    def feats(keys):
        t = [str(df.get(k, "")) for k in keys]
        return np.column_stack([np.log1p([len(s.encode("utf-8")) for s in t]),
                                np.log1p([len(s.split()) for s in t])])
    Xtr, Xte = feats(train_keys), feats(eval_keys)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    clf = LogisticRegression(max_iter=3000).fit((Xtr-mu)/sd, y_train)
    p = clf.predict((Xte-mu)/sd)
    len_rows.append({"language": lang,
                     "len_macro_f1": round(float(f1_score(y_eval, p, average="macro",
                                                          zero_division=0)), 4)})
lenp = pd.DataFrame(len_rows)
lenp.to_csv(f"{OUT}/probe2_length_baseline.csv", index=False)
print(f"length-only macro-F1: mean {lenp.len_macro_f1.mean():.3f}  "
      f"SD {lenp.len_macro_f1.std():.3f}  max {lenp.len_macro_f1.max():.3f}")
print(f"encoder probe macro-F1: mean {probe2.macro_f1.mean():.3f}")

## 6. Does it agree with the binary probe?

This is the key comparison. Upload `downstream_all_encoders.csv` (the binary
probe), `retrieval_per_language.csv` and `fertility_production_v3_spm.csv`, then compare.

In [ ]:
from scipy.stats import spearmanr, rankdata, pearsonr

try:
    from google.colab import files
    need = [f for f in ["downstream_all_encoders.csv", "retrieval_per_language.csv",
                        "fertility_production_v3_spm.csv"] if not os.path.exists(f)]
    if need:
        print("upload:", need); files.upload()
except Exception as e:
    print("no upload dialog:", e)

probe1 = pd.read_csv("downstream_all_encoders.csv")
retr   = pd.read_csv("retrieval_per_language.csv")
fert   = pd.read_csv("fertility_production_v3_spm.csv")

ENCMAP = {"sentence-transformers/LaBSE": "labse", "BAAI/bge-m3": "bgem3",
          "intfloat/multilingual-e5-base": "e5",
          "sentence-transformers/paraphrase-multilingual-mpnet-base-v2": "mpnet"}
wide = (retr.assign(e=retr.model.map(ENCMAP)).dropna(subset=["e"])
            .pivot(index="language", columns="e", values="mean_top1"))

def partial_spearman(x, y, z):
    rx, ry, rz = (rankdata(v) for v in (x, y, z))
    Z = np.column_stack([np.ones(len(rz)), rz])
    def resid(a):
        b, *_ = np.linalg.lstsq(Z, a, rcond=None); return a - Z @ b
    return pearsonr(resid(rx), resid(ry))[0]

out = []
for enc in ["labse", "bgem3", "mpnet", "e5"]:
    a = probe2[probe2.encoder == enc][["language", "macro_f1"]].rename(columns={"macro_f1": "p2"})
    b = probe1[probe1.encoder == enc][["language", "macro_f1"]].rename(columns={"macro_f1": "p1"})
    s = (a.merge(b, on="language")
          .merge(wide[[enc]].rename(columns={enc: "ret"}), left_on="language", right_index=True)
          .merge(fert[["language", "gpt4o_o200k_tok_per_verse"]], on="language"))
    s = s[s.language != "bosnian-mihanovich"]
    f = s.gpt4o_o200k_tok_per_verse.values
    out.append({"encoder": enc, "n": len(s),
                "p2_mean": round(s.p2.mean(), 3),
                "rho(p2,p1)":  round(spearmanr(s.p2, s.p1)[0], 3),
                "rho(p2,ret)": round(spearmanr(s.p2, s.ret)[0], 3),
                "rho(p2,fert)": round(spearmanr(s.p2, f)[0], 3),
                "partial(p2,fert|ret)": round(partial_spearman(s.p2.values, f, s.ret.values), 3)})

agree = pd.DataFrame(out)
agree.to_csv(f"{OUT}/probe2_agreement.csv", index=False)
print(agree.to_string(index=False))

# noise floor: how much does a per-language macro-F1 move under resampling?
sd = probe2.groupby("encoder").macro_f1.std()
print("\nper-language macro-F1 spread by encoder (SD across the 41 languages):")
print(sd.round(3).to_string())
print("Compare with the ~0.016 bootstrap SD of a single language's macro-F1 at this")
print("sample size: cross-language spread must exceed that for the ordering to mean anything.")
print("\nrho(p2,p1) is the key number: do two unrelated tasks")
print("rank the 40 languages the same way?")

## 7. Package

In [ ]:
summary = {
    "task": f"{len(CLASSES)}-way sura classification",
    "tfidf_english_reference_macro_f1": 0.278,
    "n_classes": len(CLASSES), "n_eval": int(N_EVAL), "n_train": int(len(train_keys)),
    "random_baseline": round(1/len(CLASSES), 3),
    "probe2_mean_macro_f1": round(float(probe2.macro_f1.mean()), 3),
    "length_baseline_mean": round(float(lenp.len_macro_f1.mean()), 3),
    "median_rho_probe2_probe1": round(float(agree["rho(p2,p1)"].median()), 3),
    "median_rho_probe2_retrieval": round(float(agree["rho(p2,ret)"].median()), 3),
}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
!zip -qr results_probe2.zip {OUT} && echo zipped
try:
    from google.colab import files; files.download("results_probe2.zip")
except Exception as e:
    print("download from the file browser:", e)